# Fine-tune GPT-2 on Recipe Dataset

**Purpose**: Fine-tune GPT-2 Medium model on RecipeNLG dataset for better recipe generation

**Task**: T021 [P1] [US1] - Train Recipe Generation Model

**Input**: Processed recipes from `load_recipe_dataset.ipynb`

**Output**: Fine-tuned GPT-2 model saved to `models/recipe_generation/finetuned/`

**Dataset**: RecipeNLG (2.23M recipes)

**Training Time**: ~6-12 hours (GPU) / ~2-3 days (CPU)

## 1. Environment Setup

In [1]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')


# PyTorch and Transformers
import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    GPT2Config,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from torch.utils.data import Dataset

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("✅ Packages imported successfully")
print(f"   - PyTorch version: {torch.__version__}")
print(f"   - CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"   - GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✅ Packages imported successfully
   - PyTorch version: 2.7.1+cu118
   - CUDA available: True
   - CUDA device: NVIDIA GeForce RTX 3060 Laptop GPU
   - GPU memory: 6.4 GB


## 2. Configure Paths and Parameters

In [2]:
# Project directories
PROJECT_ROOT = Path.cwd().parent.parent
MODEL_DIR = PROJECT_ROOT / "models" / "recipe_generation"
FINETUNED_MODEL_DIR = MODEL_DIR / "finetuned"
CACHE_DIR = PROJECT_ROOT / "models" / ".cache" / "huggingface"
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "recipes"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "recipe_generation" / "checkpoints"

# Create directories
FINETUNED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
MODEL_NAME = "gpt2-medium"
MAX_LENGTH = 512

# Training configuration
BATCH_SIZE = 4  # Adjust based on GPU memory (4 for 8GB, 8 for 16GB)
GRADIENT_ACCUMULATION = 4  # Effective batch size = 4 * 4 = 16
LEARNING_RATE = 5e-5
NUM_EPOCHS = 3  # 3 epochs for training
WARMUP_STEPS = 100  # Reduced for smaller dataset
SAVE_STEPS = 100    # Save checkpoint every 100 steps (was 5000)
EVAL_STEPS = 100    # Evaluate every 100 steps (was 2500)
LOGGING_STEPS = 50  # Log more frequently (was 100)

# Training subset (for testing, set to None for full dataset)
TRAIN_SUBSET = 10000  # Set to 10000 for quick test, None for full training

print(f"📁 Model directory: {MODEL_DIR}")
print(f"📁 Fine-tuned model: {FINETUNED_MODEL_DIR}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"📁 Checkpoints: {CHECKPOINT_DIR}")
print(f"\n🎯 Training Configuration:")
print(f"   - Base model: {MODEL_NAME}")
print(f"   - Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * GRADIENT_ACCUMULATION})")
print(f"   - Learning rate: {LEARNING_RATE}")
print(f"   - Epochs: {NUM_EPOCHS}")
print(f"   - Max length: {MAX_LENGTH}")
print(f"   - Training subset: {TRAIN_SUBSET or 'Full dataset'}")
print(f"   - Save/Eval every: {SAVE_STEPS} steps")
print(f"   - Warmup steps: {WARMUP_STEPS}")

📁 Model directory: c:\Users\Champion\Documents\GitHub\cAIuldron\models\recipe_generation
📁 Fine-tuned model: c:\Users\Champion\Documents\GitHub\cAIuldron\models\recipe_generation\finetuned
📁 Data directory: c:\Users\Champion\Documents\GitHub\cAIuldron\data\processed\recipes
📁 Checkpoints: c:\Users\Champion\Documents\GitHub\cAIuldron\models\recipe_generation\checkpoints

🎯 Training Configuration:
   - Base model: gpt2-medium
   - Batch size: 4 (effective: 16)
   - Learning rate: 5e-05
   - Epochs: 3
   - Max length: 512
   - Training subset: 10000
   - Save/Eval every: 100 steps
   - Warmup steps: 100


## 3. Load Pre-trained Model and Tokenizer

In [3]:
# Set cache directory
os.environ['TRANSFORMERS_CACHE'] = str(CACHE_DIR)

print("📥 Loading pre-trained GPT-2 model...\n")

# Load tokenizer
print("1️⃣ Loading tokenizer...")
tokenizer = GPT2Tokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=CACHE_DIR
)

# Add padding token
tokenizer.pad_token = tokenizer.eos_token

print(f"   ✅ Tokenizer loaded")
print(f"   - Vocabulary size: {len(tokenizer)}")

# Load model
print("\n2️⃣ Loading GPT-2 model...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   - Device: {device}")

model = GPT2LMHeadModel.from_pretrained(
    MODEL_NAME,
    cache_dir=CACHE_DIR
)

# Resize token embeddings if needed
model.resize_token_embeddings(len(tokenizer))

print(f"   ✅ Model loaded")
print(f"   - Parameters: {model.num_parameters():,}")
print(f"   - Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

print("\n✅ Pre-trained model ready for fine-tuning!")

📥 Loading pre-trained GPT-2 model...

1️⃣ Loading tokenizer...
   ✅ Tokenizer loaded
   - Vocabulary size: 50257

2️⃣ Loading GPT-2 model...
   - Device: cuda
   ✅ Model loaded
   - Parameters: 354,823,168
   - Trainable parameters: 354,823,168

✅ Pre-trained model ready for fine-tuning!


## 4. Load Recipe Dataset

In [4]:
# Load processed recipes
recipe_file = DATA_DIR / "full_recipes.json"

if not recipe_file.exists():
    print("❌ Recipe dataset not found!")
    print(f"   Expected: {recipe_file}")
    print("\n📝 Please run load_recipe_dataset.ipynb first to process the dataset.")
    raise FileNotFoundError(f"Dataset not found: {recipe_file}")

print(f"📂 Loading recipes from: {recipe_file}")
with open(recipe_file, 'r', encoding='utf-8') as f:
    recipes = json.load(f)

print(f"✅ Loaded {len(recipes):,} recipes")

# Apply training subset if specified
if TRAIN_SUBSET:
    recipes = recipes[:TRAIN_SUBSET]
    print(f"⚠️ Using subset of {len(recipes):,} recipes for training")

# Check data format
print(f"\n📋 Sample recipe structure:")
sample = recipes[0]
print(f"   Keys: {list(sample.keys())}")
print(f"\n   Example:")
print(f"   - Ingredient: {sample['ingredient']}")
print(f"   - Title: {sample['recipe_title']}")
print(f"   - Cuisine: {sample['cuisine']}")
print(f"   - Ingredients: {len(sample['ingredients'])} items")
print(f"   - Instructions: {len(sample['instructions'])} steps")

📂 Loading recipes from: c:\Users\Champion\Documents\GitHub\cAIuldron\data\processed\recipes\full_recipes.json
✅ Loaded 7,913 recipes
⚠️ Using subset of 7,913 recipes for training

📋 Sample recipe structure:
   Keys: ['Unnamed: 0', 'recipe_title', 'ingredients', 'instructions', 'link', 'source', 'ingredient_tags', 'ingredient', 'cuisine', 'difficulty', 'servings', 'cooking_time_minutes', 'formatted_text']

   Example:
   - Ingredient: c.
   - Title: No-Bake Nut Cookies
   - Cuisine: american
   - Ingredients: 6 items
   - Instructions: 6 steps


## 5. Create Custom Dataset Class

In [5]:
class RecipeDataset(Dataset):
    """
    Custom dataset for recipe generation training
    """
    
    def __init__(self, recipes, tokenizer, max_length=512):
        self.recipes = recipes
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.recipes)
    
    def __getitem__(self, idx):
        recipe = self.recipes[idx]
        
        # Format recipe as structured text
        text = self.format_recipe(recipe)
        
        # Tokenize
        encodings = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        # Return input_ids and labels (same for language modeling)
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze(),
            'labels': encodings['input_ids'].squeeze()
        }
    
    def format_recipe(self, recipe):
        """
        Format recipe as structured text with tags
        """
        # Ingredient tags
        ingredients_text = "; ".join(recipe.get('ingredients', []))
        
        # Instruction tags
        instructions_text = " ".join([
            f"{i+1}. {step}"
            for i, step in enumerate(recipe.get('instructions', []))
        ])
        
        # Format with tags
        text = (
            f"<INGREDIENT> {recipe.get('ingredient', 'unknown')}\n"
            f"<TITLE> {recipe.get('recipe_title', 'Untitled')}\n"
            f"<CUISINE> {recipe.get('cuisine', 'Unknown')}\n"
            f"<DIFFICULTY> {recipe.get('difficulty', 'medium')}\n"
            f"<TIME> {recipe.get('cooking_time_minutes', 30)}\n"
            f"<SERVINGS> {recipe.get('servings', 2)}\n"
            f"<INGREDIENTS> {ingredients_text}\n"
            f"<INSTRUCTIONS> {instructions_text}\n"
        )
        
        return text

print("✅ RecipeDataset class defined")

# Test dataset
print("\n🧪 Testing dataset...")
test_dataset = RecipeDataset(recipes[:5], tokenizer, MAX_LENGTH)
sample_item = test_dataset[0]

print(f"   - Dataset size: {len(test_dataset)}")
print(f"   - Sample input_ids shape: {sample_item['input_ids'].shape}")
print(f"   - Sample attention_mask shape: {sample_item['attention_mask'].shape}")
print(f"\n   - Decoded text (first 200 chars):")
decoded = tokenizer.decode(sample_item['input_ids'][:200])
print(f"     {decoded}...")

✅ RecipeDataset class defined

🧪 Testing dataset...
   - Dataset size: 5
   - Sample input_ids shape: torch.Size([512])
   - Sample attention_mask shape: torch.Size([512])

   - Decoded text (first 200 chars):
     <INGREDIENT> c.
<TITLE> No-Bake Nut Cookies
<CUISINE> american
<DIFFICULTY> easy
<TIME> 30
<SERVINGS> 4
<INGREDIENTS> 1 c. firmly packed brown sugar; 1/2 c. evaporated milk; 1/2 tsp. vanilla; 1/2 c. broken nuts (pecans); 2 Tbsp. butter or margarine; 3 1/2 c. bite size shredded rice biscuits
<INSTRUCTIONS> 1. In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine. 2. Stir over medium heat until mixture bubbles all over top. 3. Boil and stir 5 minutes more. Take off heat. 4. Stir in vanilla and cereal; mix well. 5. Using 2 teaspoons, drop and shape into 30 clusters on wax paper. 6. Let stand until...


## 6. Split Dataset into Train/Validation

In [6]:
# Split dataset
split_ratio = 0.95  # Use 95% for training, 5% for validation
split_idx = int(len(recipes) * split_ratio)

train_recipes = recipes[:split_idx]
val_recipes = recipes[split_idx:]

print(f"📊 Dataset Split:")
print(f"   - Training: {len(train_recipes):,} recipes ({split_ratio*100:.0f}%)")
print(f"   - Validation: {len(val_recipes):,} recipes ({(1-split_ratio)*100:.0f}%)")

# Create datasets
print("\n🔨 Creating datasets...")
train_dataset = RecipeDataset(train_recipes, tokenizer, MAX_LENGTH)
val_dataset = RecipeDataset(val_recipes, tokenizer, MAX_LENGTH)

print(f"✅ Datasets created")
print(f"   - Train dataset: {len(train_dataset):,} samples")
print(f"   - Val dataset: {len(val_dataset):,} samples")

# Calculate training steps
total_steps = (len(train_dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION)) * NUM_EPOCHS
print(f"\n📈 Training Statistics:")
print(f"   - Total steps: {total_steps:,}")
print(f"   - Steps per epoch: {total_steps // NUM_EPOCHS:,}")
print(f"   - Estimated time (GPU): {total_steps * 0.5 / 3600:.1f} hours")
print(f"   - Estimated time (CPU): {total_steps * 3 / 3600:.1f} hours")

📊 Dataset Split:
   - Training: 7,517 recipes (95%)
   - Validation: 396 recipes (5%)

🔨 Creating datasets...
✅ Datasets created
   - Train dataset: 7,517 samples
   - Val dataset: 396 samples

📈 Training Statistics:
   - Total steps: 1,407
   - Steps per epoch: 469
   - Estimated time (GPU): 0.2 hours
   - Estimated time (CPU): 1.2 hours


## 7. Configure Training Arguments

In [7]:
# Training arguments
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    overwrite_output_dir=True,
    
    # Training parameters
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    
    # Optimization
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    adam_epsilon=1e-8,
    max_grad_norm=1.0,
    
    # Logging and saving
    logging_dir=str(CHECKPOINT_DIR / "logs"),
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,  # Keep only 3 best checkpoints
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Performance
    fp16=torch.cuda.is_available(),  # Use mixed precision on GPU
    dataloader_num_workers=0,  # Set to 0 to avoid multiprocessing issues
    
    # Reporting
    report_to="none",  # Disable wandb/tensorboard for simplicity
    
    # Other
    seed=42,
    push_to_hub=False
)

print("✅ Training arguments configured")
print(f"\n📋 Key Settings:")
print(f"   - Output: {training_args.output_dir}")
print(f"   - Epochs: {training_args.num_train_epochs}")
print(f"   - Batch size: {training_args.per_device_train_batch_size}")
print(f"   - Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"   - Learning rate: {training_args.learning_rate}")
print(f"   - FP16: {training_args.fp16}")
print(f"   - Save steps: {training_args.save_steps}")
print(f"   - Eval steps: {training_args.eval_steps}")

✅ Training arguments configured

📋 Key Settings:
   - Output: c:\Users\Champion\Documents\GitHub\cAIuldron\models\recipe_generation\checkpoints
   - Epochs: 3
   - Batch size: 4
   - Gradient accumulation: 4
   - Learning rate: 5e-05
   - FP16: True
   - Save steps: 100
   - Eval steps: 100


## 8. Create Trainer

In [8]:
# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # GPT-2 uses causal language modeling, not masked LM
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("✅ Trainer created")
print(f"\n🎯 Ready to start training!")
print(f"\n⚠️ Important:")
print(f"   - Training will take several hours")
print(f"   - GPU recommended for faster training")
print(f"   - Checkpoints saved every {SAVE_STEPS:,} steps")
print(f"   - You can resume training from checkpoints if interrupted")

✅ Trainer created

🎯 Ready to start training!

⚠️ Important:
   - Training will take several hours
   - GPU recommended for faster training
   - Checkpoints saved every 100 steps
   - You can resume training from checkpoints if interrupted


## 9. Start Training

In [9]:
print("🚀 Starting training...\n")
print("=" * 70)

# Train the model
train_result = trainer.train()

print("\n" + "=" * 70)
print("\n✅ Training completed!")
print(f"\n📊 Training Results:")
print(f"   - Final loss: {train_result.training_loss:.4f}")
print(f"   - Total steps: {train_result.global_step}")
print(f"   - Training time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"   - Samples/second: {train_result.metrics['train_samples_per_second']:.2f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


🚀 Starting training...



`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,1.428100,1.320742
200,1.341900,1.255001
300,1.296000,1.224907
400,1.267200,1.199467
500,1.214300,1.182667
600,1.195200,1.176589
700,1.161100,1.161083
800,1.178100,1.150261
900,1.169200,1.143032


KeyboardInterrupt: 

## 10. Evaluate Model

In [ ]:
print("📊 Evaluating model on validation set...\n")

eval_results = trainer.evaluate()

print("✅ Evaluation completed!")
print(f"\n📈 Evaluation Metrics:")
print(f"   - Validation loss: {eval_results['eval_loss']:.4f}")
print(f"   - Perplexity: {np.exp(eval_results['eval_loss']):.2f}")

📊 Evaluating model on validation set...



## 11. Save Fine-tuned Model

In [ ]:
print(f"💾 Saving fine-tuned model to: {FINETUNED_MODEL_DIR}\n")

# Save model and tokenizer
trainer.save_model(str(FINETUNED_MODEL_DIR))
tokenizer.save_pretrained(str(FINETUNED_MODEL_DIR))

print("✅ Model saved successfully!")
print(f"\n📁 Saved files:")
saved_files = list(FINETUNED_MODEL_DIR.glob('*'))
for f in saved_files:
    size_mb = f.stat().st_size / (1024**2)
    print(f"   - {f.name}: {size_mb:.1f} MB")

# Save training metadata
metadata = {
    'model_name': MODEL_NAME,
    'base_model': MODEL_NAME,
    'training_data': 'RecipeNLG',
    'num_recipes': len(recipes),
    'num_train': len(train_recipes),
    'num_val': len(val_recipes),
    'epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
    'gradient_accumulation': GRADIENT_ACCUMULATION,
    'learning_rate': LEARNING_RATE,
    'max_length': MAX_LENGTH,
    'final_train_loss': float(train_result.training_loss),
    'final_eval_loss': float(eval_results['eval_loss']),
    'perplexity': float(np.exp(eval_results['eval_loss'])),
    'training_time_seconds': train_result.metrics['train_runtime'],
    'device': str(device),
    'status': 'fine-tuned'
}

metadata_file = FINETUNED_MODEL_DIR / 'training_metadata.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ Metadata saved to: {metadata_file}")

## 12. Test Fine-tuned Model

In [ ]:
print("🧪 Testing fine-tuned model...\n")

# Test generation
test_ingredient = "chicken breast"
test_prompt = f"<INGREDIENT> {test_ingredient}\n<TITLE>"

print(f"Test prompt: '{test_prompt}'")
print("\n" + "="*70)

# Encode and generate
input_ids = tokenizer.encode(test_prompt, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_length=300,
        temperature=0.9,
        top_k=50,
        top_p=0.95,
        num_return_sequences=2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=2
    )

# Display results
for i, output in enumerate(outputs, 1):
    generated_text = tokenizer.decode(output, skip_special_tokens=True)
    print(f"\n📝 Generated Recipe {i}:")
    print(generated_text)
    print("\n" + "-"*70)

print("\n✅ Test generation completed!")

## 13. Summary

### ✅ Completed:
1. ✅ Loaded pre-trained GPT-2 Medium
2. ✅ Prepared RecipeNLG dataset for training
3. ✅ Created custom dataset class with structured formatting
4. ✅ Split into train/validation sets
5. ✅ Configured training arguments
6. ✅ Fine-tuned model on recipe data
7. ✅ Evaluated model performance
8. ✅ Saved fine-tuned model
9. ✅ Tested generation quality

### 🎯 Training Results:
- **Dataset**: RecipeNLG (2.23M recipes)
- **Model**: GPT-2 Medium (355M parameters)
- **Epochs**: 3
- **Final Loss**: See cell output above
- **Perplexity**: See cell output above

### 📁 Output:
- **Fine-tuned model**: `models/recipe_generation/finetuned/`
- **Checkpoints**: `models/recipe_generation/checkpoints/`
- **Metadata**: `training_metadata.json`

### 📝 Next Steps:
1. **Update `model_gpt2_inference.ipynb`** to use fine-tuned model
2. **Test generation quality** on various ingredients
3. **Compare** pre-trained vs fine-tuned performance
4. **Integrate** with full pipeline (ingredient recognition → recipes)

### 💡 Usage:
```python
# Load fine-tuned model
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_path = "models/recipe_generation/finetuned"
model = GPT2LMHeadModel.from_pretrained(model_path)
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
```

### ⚠️ Notes:
- Training time varies by hardware (6-12h on GPU, 2-3 days on CPU)
- Can resume from checkpoints if interrupted
- Fine-tuned model should generate better, more realistic recipes
- Perplexity lower = better model performance

In [ ]:
print("🎉 Fine-tuning Complete!")
print(f"\n📁 Fine-tuned model: {FINETUNED_MODEL_DIR}")
print(f"\n✅ Ready to use in recipe generation pipeline!")
print(f"\n💡 Next: Update model_gpt2_inference.ipynb to use this model")